# 09. 리뷰 VOC 2계층 재분류 (도메인 분리)

## 배경
초기 키워드 분류는 충전 7개 카테고리만 두어 **부정 리뷰의 50%가 '기타'**로 빠졌다.
원인 진단(DB 표본 분석): `app_reviews`의 **92%가 Green SM**(베트남 EV '라이드헤일링' 앱 *Xanh SM*)이라,
충전과 무관한 **배차·드라이버·광고** 불만이 대부분이어서 충전 카테고리에 안 맞고 기타로 떨어진 것.

## 해결
(1) 라이드헤일링 (2) 충전 (3) 앱 공통 **3개 도메인**으로 분리하고,
vi/th/en/ko **다국어 키워드를 5차례 보강** → **기타 50.0% → 16.6%**.
분류 규칙은 `src/review_classifier.py`(재현 가능 모듈)로 분리.

> 이 노트북은 그동안 버전관리에 없던 분류 코드를 복원해 **재현성 결함도 함께 해결**한다.

In [ ]:
import sys; sys.path.insert(0, '../src'); sys.path.insert(0, 'src')
import pandas as pd
from db import get_connection
from review_classifier import classify, CATEGORY_DOMAIN
from psycopg2.extras import execute_values

## 1. 도메인 컬럼 추가 + 전체 리뷰 로드

In [ ]:
conn = get_connection(); cur = conn.cursor()
cur.execute("ALTER TABLE app_reviews ADD COLUMN IF NOT EXISTS keyword_domain VARCHAR(12);")
conn.commit()
df = pd.read_sql("SELECT id, content, sentiment_label FROM app_reviews", conn)
print(len(df), "건 로드")

## 2. 분류 (긍정오분류는 부정 리뷰에서만 의미)

In [ ]:
def cls_row(r):
    cat, dom = classify(r['content'])
    if cat == '긍정오분류' and r['sentiment_label'] != 'negative':
        return ('기타·미상', '미상')
    return (cat, dom)
res = df.apply(cls_row, axis=1)
df['cat'] = [x[0] for x in res]; df['dom'] = [x[1] for x in res]

## 3. 배치 UPDATE (Supabase 10초 timeout 회피 — 200건 단위)

In [ ]:
rows = list(zip(df.id.tolist(), df.cat.tolist(), df.dom.tolist()))
B = 200; tot = 0
for i in range(0, len(rows), B):
    execute_values(cur,
        "UPDATE app_reviews AS a SET keyword_category=v.cat, keyword_domain=v.dom "
        "FROM (VALUES %s) AS v(id,cat,dom) WHERE a.id=v.id",
        rows[i:i+B], template="(%s,%s,%s)")
    conn.commit(); tot += len(rows[i:i+B])
print("UPDATE", tot)

## 4. 검증 — 부정 리뷰 2계층 분포

In [ ]:
q = '''SELECT CASE WHEN app_name ILIKE '%green%' OR app_name ILIKE '%gsm%'
          THEN 'GreenSM(라이드헤일링)' ELSE '충전앱(PTT/PEA/EleXA/LOCA)' END grp,
          keyword_domain dom, keyword_category cat, COUNT(*) n
        FROM app_reviews WHERE sentiment_label='negative' GROUP BY 1,2,3'''
d = pd.read_sql(q, conn)
print("전체 도메인:\n", d.groupby('dom').n.sum().sort_values(ascending=False))
for g in d.grp.unique():
    sub = d[d.grp==g]; n = sub.n.sum()
    print(f"\n[{g}] 부정 {n}건")
    print((sub.groupby('cat').n.sum().sort_values(ascending=False)/n*100).round(1).head(8))
cur.close(); conn.close()

## 5. 핵심 결론
- **부정 리뷰 도메인**: 라이드헤일링 42.4% · 앱공통 39.2% · 기타 16.6% · 충전 전용 0.7%
- **충전앱 레이어**(우리 팀 담당, 부정 1,227건): 앱 불안정 **27.6%** · 계정·인증·KYC **15.0%** · 결제·지갑 **11.0%**
  → 충전앱의 진짜 우선순위는 OCPP·하드웨어가 아니라 **앱 안정성 + 온보딩 + 지갑결제**.
- **라이드헤일링 레이어**(Green SM=Xanh SM, 부정 14,234건): 배차 **21.4%** · 드라이버 12.0%
  → KOKKOK Move(라오스 현지 라이드헤일링)가 직면할 UX 시사점.
- 시각화: `outputs/18_complaint_2layer.png`